# Notebook 05 - Full Deployment Test
## Human Intrusion Detection System
End-to-end test: pipeline, all conditions, API, live webcam.
**GPU**: NVIDIA Quadro T2000 | **Classes**: person only


In [ ]:
# Fix: Robust root detection (works from any Jupyter launch directory)
import sys
from pathlib import Path
def _find_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / 'src').is_dir() and (p / 'requirements.txt').exists():
            return p
    return Path.cwd()
ROOT = _find_root()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')
print(f'src/ found  : {(ROOT / "src").is_dir()}')

import os, json, time, warnings, subprocess, threading
from datetime import datetime
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## Section 1 - Initialize Pipeline

In [ ]:
from src.models.detector import IntrusionDetector, Detection
from src.models.tracker import ByteTracker
from src.inference.zone_manager import ZoneManager
from src.inference.alert_engine import AlertEngine

WEIGHTS_PATH = ROOT / 'weights' / 'yolov8s_intrusion.pt'
ZONES_FILE   = ROOT / 'data' / 'zones' / 'zones_config.json'
weights = str(WEIGHTS_PATH) if WEIGHTS_PATH.exists() else 'yolov8s.pt'

print('Initializing pipeline...')
detector  = IntrusionDetector(weights=weights, device=DEVICE,
                               conf_threshold=0.5, iou_threshold=0.45,
                               img_size=640, half=True)
tracker   = ByteTracker(track_thresh=0.5, match_thresh=0.8,
                         track_buffer=30, min_hits=2)
zone_mgr  = ZoneManager(config_path=str(ZONES_FILE), cooldown_seconds=0)
alert_eng = AlertEngine(redis_client=None,
                         frames_save_dir=str(ROOT / 'logs' / 'frames'))
print(f'Detector  : {Path(weights).name}')
print(f'Zones     : {len(zone_mgr.zones)} configured')


## Section 2 - Test Intrusion Conditions

In [ ]:
def make_det(cx, cy, conf=0.88):
    h = 40
    return Detection(bbox=(cx-h,cy-h,cx+h,cy+h), confidence=conf,
                     class_id=0, class_name='person')

print('Testing 4 Intrusion Conditions')
print('='*55)

# Condition 1: Zone Entry
zm1 = ZoneManager(config_path=str(ZONES_FILE), cooldown_seconds=0)
a1 = zm1.evaluate([make_det(300,400)], camera_id='CAM_01')
print(f'1. Zone Entry: {"ALERT" if a1 else "no alert"}')
if a1: print(f'   zone={a1[0]["zone"]} type={a1[0]["alert_type"]} persons={a1[0]["personCount"]}')

# Condition 2: Multiple Intruders
zm2 = ZoneManager(config_path=str(ZONES_FILE), cooldown_seconds=0)
a2 = zm2.evaluate([make_det(250,380),make_det(320,410),make_det(290,450)], camera_id='CAM_01')
print(f'2. Multiple Intruders: {"ALERT" if a2 else "no alert"}')
if a2: print(f'   persons={a2[0]["personCount"]} type={a2[0]["alert_type"]}')

# Condition 3: Restricted Hours
from datetime import datetime
zm3 = ZoneManager(config_path=str(ZONES_FILE), cooldown_seconds=0)
zone = list(zm3.zones.values())[0]
now_str = datetime.now().strftime('%H:%M')
is_restricted = zone.is_restricted_now()
print(f'3. Restricted Hours: time={now_str} window={zone.restricted_start}-{zone.restricted_end} active={is_restricted}')

print('All condition tests complete.')


## Section 3 - Validate Output JSON

In [ ]:
print('Output Format Validation')
zm_f = ZoneManager(config_path=str(ZONES_FILE), cooldown_seconds=0)
mock = []
for i,(cx,cy) in enumerate([(250,380),(310,410),(280,450)]):
    d = Detection(bbox=(cx-40,cy-40,cx+40,cy+40),
                  confidence=0.88+i*0.01, class_id=0, class_name='person')
    d.track_id = i+1
    mock.append(d)

alerts = zm_f.evaluate(mock, camera_id='CAM_01')
if alerts:
    print('Generated Alert:')
    print(json.dumps(alerts[0], indent=2, default=str))
    required = ['zone','personCount','event','alert_type','confidence','timestamp','camera_id','bboxes']
    missing = [f for f in required if f not in alerts[0]]
    print(f'Required fields present: {not missing}')
    if missing: print(f'Missing: {missing}')
else:
    print('No alert (check zone polygon matches test coords)')


## Section 4 - FastAPI Test

In [ ]:
import subprocess, time, threading
API_PORT = 8000
API_BASE = f'http://localhost:{API_PORT}'

def _start_server():
    subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'src.api.main:app',
         '--host', '0.0.0.0', '--port', str(API_PORT), '--log-level', 'warning'],
        cwd=str(ROOT)
    )

print('Starting FastAPI server...')
threading.Thread(target=_start_server, daemon=True).start()

for i in range(15):
    try:
        r = requests.get(f'{API_BASE}/health', timeout=2)
        if r.status_code == 200:
            print(f'API ready: {API_BASE}')
            print(f'Docs: {API_BASE}/docs')
            break
    except Exception:
        time.sleep(2)
        print(f'  waiting... ({i+1}/15)')
else:
    print(f'Server not responding. Start manually:')
    print(f'  cd {ROOT} && uvicorn src.api.main:app --port {API_PORT} --reload')


In [ ]:
# Test POST /api/v1/detect
frame = np.ones((720,1280,3), dtype=np.uint8) * 80
for cx,cy in [(300,400),(500,350),(700,450)]:
    cv2.rectangle(frame,(cx-40,cy-80),(cx+40,cy+80),(180,180,180),-1)
_, buf = cv2.imencode('.jpg', frame)

try:
    resp = requests.post(f'{API_BASE}/api/v1/detect',
                         files={'file':('frame.jpg',buf.tobytes(),'image/jpeg')},
                         params={'camera_id':'CAM_01'}, timeout=30)
    if resp.status_code == 200:
        r = resp.json()
        print(f'POST /detect -> {resp.status_code}')
        print(f'  personCount : {r["personCount"]}')
        print(f'  alerts      : {len(r["alerts"])}')
        if r['alerts']:
            print(f'  alert zone  : {r["alerts"][0]["zone"]}')
    else:
        print(f'Error {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    print(f'API error: {e}')


## Section 5 - Live Webcam Demo

In [ ]:
# Live Webcam Demo with Zone Overlay
# Change SOURCE to a video file path if needed:
# SOURCE = r'C:\path\to\video.mp4'
SOURCE    = 0          # 0 = laptop webcam
CAM_ID    = 'CAM_01'
MAX_FRM   = 300        # stop after N frames in notebook

zm_live    = ZoneManager(config_path=str(ZONES_FILE), cooldown_seconds=10)
trk_live   = ByteTracker(track_thresh=0.5, min_hits=2)
traj_hist  = {}
alert_log  = []
fps_times  = []
frame_cnt  = 0

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    print(f'Cannot open source: {SOURCE}')
    print('Change SOURCE to a video file path.')
else:
    print(f'Starting | Source:{SOURCE} | Camera:{CAM_ID} | Press Q to quit')
    try:
        while frame_cnt < MAX_FRM:
            ret, frame = cap.read()
            if not ret: break
            t0 = time.perf_counter()
            frame_disp = cv2.resize(frame, (1280,720))
            result = detector.detect(frame_disp, camera_id=CAM_ID)
            result = trk_live.update(result)
            for det in result.detections:
                if det.track_id:
                    traj_hist.setdefault(det.track_id,[]).append(det.center)
                    if len(traj_hist[det.track_id])>30: traj_hist[det.track_id].pop(0)
            alerts = zm_live.evaluate(result.detections, CAM_ID,
                                       trajectory_history=traj_hist)
            if alerts: alert_log.extend(alerts)
            vis = zm_live.draw_zones(frame_disp, CAM_ID)
            for det in result.detections:
                x1,y1,x2,y2 = det.bbox
                c = (0,80,255) if any(a for a in alerts) else (0,255,120)
                cv2.rectangle(vis,(x1,y1),(x2,y2),c,2)
                lbl = f'#{det.track_id} {det.confidence:.2f}' if det.track_id else f'{det.confidence:.2f}'
                cv2.putText(vis,lbl,(x1,max(y1-5,10)),cv2.FONT_HERSHEY_SIMPLEX,0.5,c,1)
                if det.track_id and det.track_id in traj_hist:
                    traj = traj_hist[det.track_id]
                    for i in range(1,len(traj)):
                        cv2.line(vis,traj[i-1],traj[i],(0,200,80),2)
            if alerts:
                cv2.rectangle(vis,(0,0),(vis.shape[1],55),(0,0,200),-1)
                cv2.putText(vis,f'ALERT: {alerts[0]["alert_type"]} | {alerts[0]["zone"]}',
                            (10,38),cv2.FONT_HERSHEY_SIMPLEX,0.9,(255,255,255),2)
            fps_times.append(time.perf_counter()-t0)
            fps = 1/(np.mean(fps_times[-30:])+1e-6)
            cv2.putText(vis,f'FPS:{fps:.1f} | Persons:{result.person_count}',
                        (10,vis.shape[0]-15),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,0),2)
            cv2.imshow('Intrusion Detection - Q to quit', vis)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
            frame_cnt += 1
    finally:
        cap.release(); cv2.destroyAllWindows()
        avg_fps = frame_cnt/(sum(fps_times)+1e-6)
        print(f'Done | Frames:{frame_cnt} | AvgFPS:{avg_fps:.1f} | Alerts:{len(alert_log)}')
